In [ ]:
# ============================================================
# SPEECH & TEXT-BASED FAKE REVIEW DETECTION SYSTEM
# Google Colab Browser Application
# ============================================================

!pip -q install flask scikit-learn textblob

import re
import threading
import time

from collections import Counter

from flask import Flask, request, jsonify, render_template_string

from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from google.colab.output import eval_js

# Initialize the Flask application
app = Flask(__name__)

# ============================================================
# 1. TRAINING DATA
# Demo dataset for educational/project prototype
# ============================================================

real_reviews = [

    "The phone works well and the battery lasts about a day.",
    "The delivery arrived two days late but the product was fine.",
    "I have been using this laptop for three months and the keyboard is comfortable.",
    "The food was good but the portion was smaller than expected.",
    "The headphones have clear sound and are comfortable for long calls.",
    "The package arrived safely and everything was properly packed.",
    "The camera quality is decent in daylight but not very good at night.",
    "The shirt fits well and the material is comfortable.",
    "The hotel room was clean but the bathroom was quite small.",
    "The product is useful and works as described.",
    "The battery lasts around eight hours with normal usage.",
    "The delivery was quick and the packaging was okay.",
    "The shoes are comfortable but the size runs slightly small.",
    "The screen is bright and the speakers are acceptable.",
    "I bought this for college and it works well for my daily tasks."

]


fake_reviews = [

    "AMAZING AMAZING AMAZING!!! BEST PRODUCT EVER!!! BUY NOW!!!",
    "This is absolutely perfect and everyone should buy this immediately!!!",
    "Five stars!!! Best thing I have ever purchased in my entire life!!!",
    "WOW WOW WOW this product changed my life completely!!!",
    "Perfect perfect perfect!!! Nothing could possibly be better!!!",
    "BEST PRODUCT ON THE INTERNET!!! MUST BUY!!!",
    "I cannot believe how amazing this is!!! Absolutely flawless!!!",
    "Everyone needs this product!!! It is the greatest thing ever!!!",
    "This product is unbelievably perfect!!! Buy it right now!!!",
    "Amazing quality amazing service amazing experience five stars!!!",
    "I LOVE THIS SO MUCH!!! BEST PURCHASE EVER!!!",
    "This is the greatest product ever created!!! Highly highly recommended!!!",
    "Perfect product!!! Perfect delivery!!! Perfect quality!!! Five stars!!!",
    "Buy this immediately because it is absolutely incredible!!!",
    "WOW!!! AMAZING!!! PERFECT!!! BEST!!! HIGHLY RECOMMENDED!!!"

]


training_texts = real_reviews + fake_reviews

training_labels = (
    [0] * len(real_reviews) +
    [1] * len(fake_reviews)
)


# ============================================================
# 2. MACHINE LEARNING MODEL
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(training_texts)

model = LogisticRegression(
    max_iter=1000
)

model.fit(X, training_labels)


# ============================================================
# 3. TEXT ANALYSIS
# ============================================================

def analyze_review(text):

    text = text.strip()

    if not text:

        return {
            "error": "Please enter or speak a review."
        }


    # --------------------------------------------------------
    # ML prediction
    # --------------------------------------------------------

    vector = vectorizer.transform([text])

    probability = model.predict_proba(vector)[0][1]

    ml_score = probability * 100


    # --------------------------------------------------------
    # Basic text statistics
    # --------------------------------------------------------

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text.lower()
    )

    word_count = len(words)

    character_count = len(text)

    sentence_count = len(
        re.findall(
            r"[.!?]+",
            text
        )
    )


    # --------------------------------------------------------
    # Sentiment
    # --------------------------------------------------------

    blob = TextBlob(text)

    polarity = blob.sentiment.polarity

    if polarity > 0.1:

        sentiment = "Positive 😊"

    elif polarity < -0.1:

        sentiment = "Negative 😞"

    else:

        sentiment = "Neutral 😐"


    # --------------------------------------------------------
    # Suspicious patterns
    # --------------------------------------------------------

    reasons = []

    suspicious_score = ml_score


    # Excessive exclamation marks
    exclamation_count = text.count("!")

    if exclamation_count >= 3:

        suspicious_score += 10

        reasons.append(
            "Excessive use of exclamation marks"
        )


    # Repeated words
    word_frequency = Counter(words)

    repeated_words = [
        word
        for word, count
        in word_frequency.items()
        if count >= 3 and len(word) > 2
    ]

    if repeated_words:

        suspicious_score += 10

        reasons.append(
            "Repeated words detected"
        )


    # Promotional words
    promotional_words = [
        "buy now",
        "must buy",
        "best ever",
        "perfect",
        "amazing",
        "greatest",
        "five stars",
        "highly recommended",
        "everyone should buy"
    ]

    promotional_found = []

    lower_text = text.lower()

    for phrase in promotional_words:

        if phrase in lower_text:

            promotional_found.append(
                phrase
            )


    if promotional_found:

        suspicious_score += 10

        reasons.append(
            "Strong promotional language detected"
        )


    # Excessive capital letters
    letters = [
        char
        for char in text
        if char.isalpha()
    ]

    if len(letters) > 10:

        uppercase_letters = [
            char
            for char in letters
            if char.isupper()
        ]

        uppercase_ratio = (
            len(uppercase_letters)
            / len(letters)
        )

        if uppercase_ratio > 0.35:

            suspicious_score += 10

            reasons.append(
                "Unusually high use of capital letters"
            )


    # Very short review
    if word_count < 5:

        suspicious_score += 5

        reasons.append(
            "Review contains very little information"
        )


    # Very extreme positive language
    extreme_words = [
        "perfect",
        "greatest",
        "best",
        "incredible",
        "flawless",
        "amazing"
    ]

    extreme_count = sum(
        lower_text.count(word)
        for word in extreme_words
    )

    if extreme_count >= 2:

        suspicious_score += 10

        reasons.append(
            "Highly exaggerated language detected"
        )


    # Keep score between 0 and 100

    suspicious_score = min(
        100,
        round(suspicious_score, 1)
    )


    # --------------------------------------------------------
    # Final classification
    # --------------------------------------------------------

    if suspicious_score >= 70:

        classification = "Highly Suspicious ⚠️"

    elif suspicious_score >= 45:

        classification = "Possibly Suspicious 🔎"

    else:

        classification = "Likely Genuine ✅"


    # --------------------------------------------------------
    # Keywords
    # --------------------------------------------------------

    stop_words = {
        "the", "is", "a", "an",
        "and", "or", "to", "of",
        "in", "on", "for", "with",
        "this", "that", "it",
        "i", "you", "was",
        "were", "very"
    }

    keywords = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 3
    ]

    keyword_frequency = Counter(
        keywords
    )

    top_keywords = [
        word
        for word, count
        in keyword_frequency.most_common(5)
    ]


    if not reasons:

        reasons.append(
            "No major suspicious patterns detected"
        )


    return {

        "classification":
            classification,

        "score":
            suspicious_score,

        "ml_score":
            round(ml_score, 1),

        "sentiment":
            sentiment,

        "polarity":
            round(polarity, 2),

        "word_count":
            word_count,

        "character_count":
            character_count,

        "sentence_count":
            sentence_count,

        "keywords":
            top_keywords,

        "reasons":
            reasons
    }


# ============================================================
# 4. HTML APPLICATION
# ============================================================

HTML = r"""

<!DOCTYPE html>

<html>

<head>

<title>
AI Review Authenticity Analyzer
</title>

<meta name="viewport"
content="width=device-width, initial-scale=1.0">


<style>

* {
    box-sizing: border-box;
}


body {

    margin: 0;

    font-family: Arial, sans-serif;

    min-height: 100vh;

    padding: 25px;

    background:
    linear-gradient(
        135deg,
        #0f172a,
        #1e3a8a,
        #312e81
    );

    color: white;
}


.container {

    max-width: 1000px;

    margin: auto;

    background:
    rgba(255,255,255,0.10);

    backdrop-filter:
    blur(15px);

    border:
    1px solid
    rgba(255,255,255,0.2);

    border-radius: 25px;

    padding: 30px;

    box-shadow:
    0 20px 60px
    rgba(0,0,0,0.3);
}


h1 {

    text-align: center;

    margin-bottom: 5px;

    font-size: 34px;
}


.subtitle {

    text-align: center;

    color: #dbeafe;

    margin-bottom: 30px;
}


textarea {

    width: 100%;

    height: 180px;

    border: none;

    border-radius: 15px;

    padding: 18px;

    font-size: 16px;

    resize: vertical;

    outline: none;
}


.buttons {

    display: flex;

    gap: 10px;

    margin-top: 15px;

    flex-wrap: wrap;
}


button {

    border: none;

    border-radius: 10px;

    padding: 13px 20px;

    cursor: pointer;

    font-size: 15px;

    font-weight: bold;
}


.voice {

    background: #111827;

    color: white;
}


.analyze {

    background: #2563eb;

    color: white;
}


.clear {

    background: #dc2626;

    color: white;
}


button:hover {

    opacity: 0.85;
}


#status {

    margin-top: 15px;

    color: #dbeafe;

    font-weight: bold;
}


.dashboard {

    margin-top: 30px;
}


.cards {

    display: grid;

    grid-template-columns:
    repeat(4, 1fr);

    gap: 15px;

    margin-top: 20px;
}


.card {

    background:
    rgba(255,255,255,0.12);

    border-radius: 15px;

    padding: 20px;

    text-align: center;
}


.card h3 {

    font-size: 14px;

    color: #cbd5e1;
}


.card p {

    font-size: 22px;

    font-weight: bold;

    margin: 8px 0 0;
}


.result {

    margin-top: 20px;

    padding: 25px;

    border-radius: 15px;

    background:
    rgba(0,0,0,0.20);
}


.score-container {

    margin-top: 15px;
}


.progress {

    width: 100%;

    height: 22px;

    background: #334155;

    border-radius: 20px;

    overflow: hidden;
}


.progress-bar {

    height: 100%;

    width: 0%;

    background: #ef4444;

    transition: width 0.6s;
}


.keywords {

    display: flex;

    flex-wrap: wrap;

    gap: 8px;

    margin-top: 10px;
}


.keyword {

    background: #2563eb;

    padding: 8px 12px;

    border-radius: 20px;

    font-size: 13px;
}


.reason {

    background:
    rgba(255,255,255,0.08);

    padding: 10px;

    margin-top: 8px;

    border-radius: 8px;
}


@media(max-width:750px) {

    .cards {

        grid-template-columns:
        repeat(2, 1fr);
    }

    body {

        padding: 10px;
    }

    .container {

        padding: 20px;
    }
}

</style>

</head>


<body>


<div class="container">


<h1>
🔍 AI Review Authenticity Analyzer
</h1>


<p class="subtitle">

Speech & Text-Based Fake Review Detection System

</p>


<textarea
id="review"
placeholder="Type a product review here or click Start Speaking...">
</textarea>


<div class="buttons">


<button
class="voice"
onclick="startSpeech()">

🎤 Start Speaking

</button>


<button
class="analyze"
onclick="analyzeReview()">

🤖 Analyze Review

</button>


<button
class="clear"
onclick="clearAll()">

🗑️ Clear

</button>


</div>


<p id="status"></p>


<div class="dashboard">


<h2>
📊 Analysis Dashboard
</h2>


<div class="cards">


<div class="card">

<h3>Prediction</h3>

<p id="classification">
-
</p>

</div>


<div class="card">

<h3>Suspiciousness</h3>

<p id="score">
0%
</p>

</div>


<div class="card">

<h3>Sentiment</h3>

<p id="sentiment">
-
</p>

</div>


<div class="card">

<h3>Words</h3>

<p id="words">
0
</p>

</div>


</div>


<div class="result">


<h3>
⚠️ Suspiciousness Score
</h3>


<div class="progress">

<div
id="progressBar"
class="progress-bar">
</div>

</div>


<p id="mlScore">
ML model score: -
</p>


</div>


<div class="result">


<h3>
🔑 Important Keywords
</h3>


<div
id="keywords"
class="keywords">

No keywords yet

</div>


</div>


<div class="result">


<h3>
🧠 Why was this result produced?
</h3>


<div id="reasons">

No analysis yet

</div>


</div>


</div>


</div>


<script>


// ==========================================================
// SPEECH RECOGNITION
// ==========================================================

function startSpeech() {


    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;


    if (!SpeechRecognition) {

        document.getElementById(
            "status"
        ).innerText =
        "❌ Please use Google Chrome for speech recognition.";

        return;
    }


    const recognition =
        new SpeechRecognition();


    recognition.lang =
        "en-IN";


    recognition.continuous =
        false;


    recognition.interimResults =
        false;


    document.getElementById(
        "status"
    ).innerText =
        "🎙️ Listening... Speak your review.";


    recognition.start();


    recognition.onresult =
    function(event) {


        const speech =
            event.results[0][0]
            .transcript;


        const box =
            document.getElementById(
                "review"
            );


        box.value +=
            (box.value ? " " : "")
            + speech;


        document.getElementById(
            "status"
        ).innerText =
        "✅ Speech converted to text.";

    };


    recognition.onerror =
    function(event) {

        document.getElementById(
            "status"
        ).innerText =
        "❌ Microphone error: "
        + event.error;

    };

}


// ==========================================================
// ANALYZE REVIEW
// ==========================================================

async function analyzeReview() {


    const review =
        document.getElementById(
            "review"
        ).value.trim();


    if (!review) {

        alert(
            "Please enter or speak a review first."
        );

        return;
    }


    document.getElementById(
        "status"
    ).innerText =
    "🔄 Analyzing review...";


    try {


        const response =
            await fetch(
                "/analyze",
                {

                    method: "POST",

                    headers: {
                        "Content-Type":
                        "application/json"
                    },

                    body:
                    JSON.stringify({
                        text: review
                    })

                }
            );


        const data =
            await response.json();


        if (data.error) {

            alert(data.error);

            return;
        }


        document.getElementById(
            "classification"
        ).innerText =
            data.classification;


        document.getElementById(
            "score"
        ).innerText =
            data.score + "%";


        document.getElementById(
            "sentiment"
        ).innerText =
            data.sentiment;


        document.getElementById(
            "words"
        ).innerText =
            data.word_count;


        document.getElementById(
            "progressBar"
        ).style.width =
            data.score + "%";


        document.getElementById(
            "mlScore"
        ).innerText =
            "ML model score: "
            + data.ml_score + "%";


        // Keywords

        const keywordBox =
            document.getElementById(
                "keywords"
            );


        keywordBox.innerHTML = "";


        if (data.keywords.length === 0) {

            keywordBox.innerText =
                "No keywords found.";

        } else {


            data.keywords.forEach(
                function(keyword) {


                    const span =
                        document.createElement(
                            "span"
                        );


                    span.className =
                        "keyword";


                    span.innerText =
                        keyword;


                    keywordBox.appendChild(
                        span
                    );

                }
            );

        }


        // Reasons

        const reasonsBox =
            document.getElementById(
                "reasons"
            );


        reasonsBox.innerHTML = "";


        data.reasons.forEach(
            function(reason) {


                const div =
                    document.createElement(
                        "div"
                    );


                div.className =
                    "reason";


                div.innerText =
                    "• " + reason;


                reasonsBox.appendChild(
                    div
                );

            }
        );


        document.getElementById(
            "status"
        ).innerText =
        "✅ Analysis completed.";


    }


    catch(error) {


        console.error(error);


        document.getElementById(
            "status"
        ).innerText =
        "❌ Unable to connect to the server.";

    }

}


// ==========================================================
// CLEAR
// ==========================================================

function clearAll() {


    document.getElementById(
        "review"
    ).value = "";


    document.getElementById(
        "classification"
    ).innerText = "-";


    document.getElementById(
        "score"
    ).innerText = "0%";


    document.getElementById(
        "sentiment"
    ).innerText = "-";


    document.getElementById(
        "words"
    ).innerText = "0";


    document.getElementById(
        "progressBar"
    ).style.width = "0%";


    document.getElementById(
        "mlScore"
    ).innerText =
    "ML model score: -";


    document.getElementById(
        "keywords"
    ).innerText =
    "No keywords yet";


    document.getElementById(
        "reasons"
    ).innerText =
    "No analysis yet";


    document.getElementById(
        "status"
    ).innerText = "";

}

</script>


</body>

</html>

"""


# ============================================================
# 5. FLASK ROUTES
# ============================================================

@app.route("/")
def home():

    return render_template_string(
        HTML
    )


@app.route(
    "/analyze",
    methods=["POST"]
)
def analyze():

    data = request.get_json()

    text = data.get(
        "text",
        ""
    )

    result = analyze_review(text)

    return jsonify(result)


# ============================================================
# 6. START FLASK
# ============================================================

def run_app():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run_app
)

thread.daemon = True

thread.start()


time.sleep(3)


# ============================================================
# 7. OPEN APPLICATION IN BROWSER
# ============================================================

url = eval_js(
    "google.colab.kernel.proxyPort(5000)"
)


print("=" * 60)

print(
    "🚀 AI REVIEW AUTHENTICITY ANALYZER STARTED"
)

print("=" * 60)

print()

print(
    "🌐 Open your application:"
)

print(url)

print()

print(
    "🎤 Use Google Chrome and allow microphone access."
)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 AI REVIEW AUTHENTICITY ANALYZER STARTED

🌐 Open your application:
https://5000-m-s-kkb-usc1c1-aphpittm4345-c.us-central1-1.prod.colab.dev

🎤 Use Google Chrome and allow microphone access.
